Firstly, let's import every package we will need in this analysis. It is required to install 
```Python
%pip install atlasopenmagic
```
in order to open ATLAS Open Data

In [ ]:
import numpy as np
import pandas as pd                 
import uproot                       # to open .root files
import awkward as ak                # to read data with uproot
#import awkward_pandas
import matplotlib.pyplot as plt     # to plot
import random                       # extract random numbers
import requests                     # for HTTP access
import aiohttp                      # HTTP client support
import atlasopenmagic as atom       # to access ATLAS Open Data directly

## Import Data
Now we can download the dataset, selecting the release of interest. It is possible to see every release using
```Python
atom.available_releases()
```
We will use the 2025 release of data taken at $\sqrt{s}=13$ TeV in p-p collisions


In [60]:
atom.set_release("2025e-13tev-beta")    # select release
skim = "GamGam"                         # select skim: events with 2 photons
random.seed(24)                         # set seed for random extractions

# get keys for every dataset: Run2 datas have key="data", MC simulations have key=numbers
all_keys = atom.available_datasets()

data_url_list = atom.get_urls("data", skim, protocol="https", cache=True)      # get list of urls for Run2 data

# get list of urls for MC simulations
mc_url_list = []
for key in all_keys:
    if key != "data":
        mc_url_list += atom.get_urls(key, skim, protocol="https", cache=True)


print(f"Number of MonteCarlo simulated datasets: {len(mc_url_list)}")
print(f"Number of Run2 datasets: {len(data_url_list)}")

Release '2025e-13tev-beta' already active with cached metadata.
Active release: 2025e-13tev-beta. (Datasets path: REMOTE)


Number of MonteCarlo simulated datasets: 373
Number of Run2 datasets: 16


Now we open data in TTrees. In order to have a way to train rapidly the model, it is possible to switch between a subset of data and the entire dataset with a boolean `bool useall`

In [3]:
useall = False      # True: use all dataset
                    # False: use a subset

In [53]:
# See names of trees and branches
print("Tree name: ", uproot.open(mc_url_list[0]).keys())
print("Branches: ", uproot.open(f"{mc_url_list[0]}:analysis").keys())

Tree name:  ['analysis;1']
Branches:  ['sig_ph', 'n_sig_ph', 'num_events', 'sum_of_weights', 'sum_of_weights_squared', 'xsec', 'kfac', 'filteff', 'TriggerMatch_DILEPTON', 'ScaleFactor_MLTRIGGER', 'ScaleFactor_PILEUP', 'ScaleFactor_FTAG', 'mcWeight', 'channelNumber', 'eventNumber', 'runNumber', 'trigML', 'trigP', 'trigDT', 'trigT', 'trigE', 'trigDM', 'trigDE', 'trigM', 'trigMET', 'ScaleFactor_BTAG', 'ScaleFactor_JVT', 'jet_n', 'jet_pt', 'jet_eta', 'jet_phi', 'jet_e', 'jet_btag_quantile', 'jet_jvt', 'largeRJet_n', 'largeRJet_pt', 'largeRJet_eta', 'largeRJet_phi', 'largeRJet_e', 'largeRJet_m', 'largeRJet_D2', 'jet_pt_jer1', 'jet_pt_jer2', 'ScaleFactor_ELE', 'ScaleFactor_MUON', 'ScaleFactor_LepTRIGGER', 'ScaleFactor_MuTRIGGER', 'ScaleFactor_ElTRIGGER', 'lep_n', 'lep_type', 'lep_pt', 'lep_eta', 'lep_phi', 'lep_e', 'lep_charge', 'lep_ptvarcone30', 'lep_topoetcone20', 'lep_z0', 'lep_d0', 'lep_d0sig', 'lep_isTightID', 'lep_isMediumID', 'lep_isLooseID', 'lep_isTightIso', 'lep_isLooseIso', 'lep_

In [ ]:
tree = "analysis"           # name of TTree in ATLAS OD
#features = ["eventNumber", "photon_n", "photon_pt", "photon_eta", "photon_phi", "photon_e",
            #"photon_ptcone20", "photon_topoetcone40", "photon_isLooseIso", "photon_isTightIso"]

features = [branch for branch in (uproot.open(f"{mc_url_list[0]}:{tree}").keys()) if "photon_" in branch]

if not useall:
        mc_ind = set(random.sample(range(len(mc_url_list)), 16))
        mc_url_list = [url for i, url in enumerate(mc_url_list) if i in mc_ind]
        


awk = uproot.concatenate([f"{url}:{tree}" for url in mc_url_list],
                         filter_name=features,
                         library="ak")

mc_data = ak.to_dataframe(awk)
              


    

#mc_data = uproot.concatenate([f"{url}:{tree}" for url in mc_url_list], library="ak")
#run_data = uproot.concatenate([f"{url}:{tree}" for url in data_url_list], library="ak")

TypeError: 'ClientResponseError' object is not subscriptable

In [57]:
for url in mc_url_list:
    print([branch for branch in (uproot.open(f"{url}:{tree}").keys()) if "photon_" in branch])

['photon_n', 'photon_pt', 'photon_eta', 'photon_phi', 'photon_e', 'photon_ptcone20', 'photon_topoetcone40', 'photon_isLooseID', 'photon_isTightID', 'photon_isLooseIso', 'photon_isTightIso', 'truth_photon_n', 'truth_photon_pt', 'truth_photon_eta', 'truth_photon_phi']
['photon_n', 'photon_pt', 'photon_eta', 'photon_phi', 'photon_e', 'photon_ptcone20', 'photon_topoetcone40', 'photon_isLooseID', 'photon_isTightID', 'photon_isLooseIso', 'photon_isTightIso', 'truth_photon_n', 'truth_photon_pt', 'truth_photon_eta', 'truth_photon_phi']
['photon_n', 'photon_pt', 'photon_eta', 'photon_phi', 'photon_e', 'photon_ptcone20', 'photon_topoetcone40', 'photon_isLooseID', 'photon_isTightID', 'photon_isLooseIso', 'photon_isTightIso', 'truth_photon_n', 'truth_photon_pt', 'truth_photon_eta', 'truth_photon_phi']
['photon_n', 'photon_pt', 'photon_eta', 'photon_phi', 'photon_e', 'photon_ptcone20', 'photon_topoetcone40', 'photon_isLooseID', 'photon_isTightID', 'photon_isLooseIso', 'photon_isTightIso', 'truth_ph

In [36]:
#mc_data = ak.to_dataframe(awk)
mc_data.columns

Index(['sig_ph', 'n_sig_ph', 'num_events', 'sum_of_weights',
       'sum_of_weights_squared', 'xsec', 'kfac', 'filteff',
       'TriggerMatch_DILEPTON', 'ScaleFactor_MLTRIGGER',
       ...
       'truth_photon_n', 'truth_photon_pt', 'truth_photon_eta',
       'truth_photon_phi', 'truth_met', 'truth_met_phi', 'met', 'met_phi',
       'met_mpx', 'met_mpy'],
      dtype='object', length=118)

In [ ]:
mc = uproot.open(f"{mc_url_list[0]}:analysis")



AttributeError: 'Model_TTree_v20' object has no attribute 'head'